# 07 - Sistemas não lineares
Vamos aprender sobre como usar o método iterativo de Newton-Raphson adaptado aos sistemas de equações, envolvendo aproximações iniciais, cálculo da matriz do Jacobiano e critério de parada, para resolução de sistemas não lineares.

Crie uma nova branch (versão) do repositório:

```bash
git branch semana7
```

Faça o checkout nessa nova branch:

```bash
git checkout semana7
```

<hr />

## Atividade 1
Escreva o método iterativo de Newton-Raphson adaptado dentro do arquivo $utils/algoritmos.py$.

In [3]:
import numpy as np
from scipy.optimize import approx_fprime


def JN(x, F, eps=1e-8):
    x = np.asarray(x, dtype=float)
    n = x.size
    Jnum = np.zeros((n, n), dtype=float)

    def fi(v):
        return F(v)[i]

    for i in range(n):
        Jnum[i, :] = approx_fprime(x, fi, epsilon=eps)

    return Jnum


def G(x, F, J):
    return x - np.linalg.inv(J(x)) @ F(x)


def GN(x, F):
    return x - np.linalg.inv(JN(x, F)) @ F(x)


def fixed_point(a, g, TOL=1e-8, iter=1000):
    x = g(a)
    i = 1
    while np.linalg.norm(x - a) > TOL and i < iter:
        a = x
        x = g(a)
        i += 1
    return x


def newton_sistemas(x0, F, J=None, tol=1e-8, max_iter=100):
    x = np.asarray(x0, dtype=float).copy()
    if J is None:
        J = lambda v: JN(v, F)

    for _ in range(max_iter):
        fx = np.asarray(F(x), dtype=float)
        if np.linalg.norm(fx, ord=np.inf) <= tol:
            return x

        jac = np.asarray(J(x), dtype=float)
        delta = np.linalg.solve(jac, -fx)
        x = x + delta

        if np.linalg.norm(delta, ord=np.inf) <= tol:
            return x

    raise RuntimeError("O método de Newton-Raphson para sistemas não convergiu.")


Encontre uma aproximação numérica para o seguinte problema não linear de três equações e três incógnitas:

$$
2x_{1} - x_{2} = \cos(x_{1})
$$

$$
- x_{1} + 2x_{2} - x_{3} = \cos(x_{2})
$$

$$
- x_{2} + x_{3} = \cos(x_{3})
$$

Partindo da seguinte aproximação inicial:

$$
x^{(0)} = (1,\,1,\,1)^{T}
$$

In [4]:
import sys
import os
import numpy as np

# Sobe um nível para a raiz do projeto e adiciona ao caminho do sistema
sys.path.append(os.path.abspath(os.path.join('..')))

from utils.algoritmos import newton_sistemas


def F(x):
    x1, x2, x3 = x
    return np.array(
        [
            2.0 * x1 - x2 - np.cos(x1),
            -x1 + 2.0 * x2 - x3 - np.cos(x2),
            -x2 + x3 - np.cos(x3),
        ],
        dtype=float,
    )


def J(x):
    x1, x2, x3 = x
    return np.array(
        [
            [2.0 + np.sin(x1), -1.0, 0.0],
            [-1.0, 2.0 + np.sin(x2), -1.0],
            [0.0, -1.0, 1.0 + np.sin(x3)],
        ],
        dtype=float,
    )


x = np.array([1.0, 1.0, 1.0], dtype=float)
sol = newton_sistemas(x, F, J, tol=1e-12)
print(sol)
print(np.linalg.norm(F(sol), ord=np.inf))


[0.95127255 1.32189759 1.44618584]
1.6653345369377348e-16


## Atividade 2
Encontre uma aproximação em cada incógnita para a solução próxima da origem do sistema:

$$
6x - 2y + e^{z} = 2
$$

$$
\sin(x) - y + z = 0
$$

$$
\sin(x) + 2y + 3z = 1
$$

**Resposta esperada:**

$$
x \approx 0.259751, \quad y \approx 0.302736, \quad z \approx 0.045896
$$

In [5]:
import numpy as np
from utils.algoritmos import newton_sistemas


def F(x):
    return np.array(
        [
            6.0 * x[0] - 2.0 * x[1] + np.exp(x[2]) - 2.0,
            np.sin(x[0]) - x[1] + x[2],
            np.sin(x[0]) + 2.0 * x[1] + 3.0 * x[2] - 1.0,
        ],
        dtype=float,
    )


def J(x):
    return np.array(
        [
            [6.0, -2.0, np.exp(x[2])],
            [np.cos(x[0]), -1.0, 1.0],
            [np.cos(x[0]), 2.0, 3.0],
        ],
        dtype=float,
    )


x0 = np.array([0.0, 0.0, 0.0], dtype=float)
sol = newton_sistemas(x0, F, J, tol=1e-12)
print(f"x ≈ {sol[0]:.6f}, y ≈ {sol[1]:.6f}, z ≈ {sol[2]:.6f}")


x ≈ 0.259751, y ≈ 0.302736, z ≈ 0.045896


## Atividade 3
Considere o problema de encontrar os pontos de interseção das curvas descritas:

$$
\frac{x^{2}}{8} + \frac{(y-1)^{2}}{5} = 1
$$

$$
\tan^{-1}(x) + x = y + y^{3}
$$

Com base no gráfico, encontre soluções aproximadas para o problema e use-as para iniciar o método de Newton-Raphson para encontrar as raízes.

**Resposta esperada:**

$$
(-1.2085435,\,-1.0216674) 
\quad \text{e} \quad 
(2.7871115,\,1.3807962)
$$

In [6]:
import numpy as np
from utils.algoritmos import newton_sistemas


def F(v):
    x, y = v
    return np.array(
        [
            (x ** 2) / 8.0 + ((y - 1.0) ** 2) / 5.0 - 1.0,
            np.arctan(x) + x - y - y ** 3,
        ],
        dtype=float,
    )


def J(v):
    x, y = v
    return np.array(
        [
            [x / 4.0, 2.0 * (y - 1.0) / 5.0],
            [1.0 / (1.0 + x ** 2) + 1.0, -(1.0 + 3.0 * y ** 2)],
        ],
        dtype=float,
    )


for guess in [np.array([-1.5, -1.0]), np.array([2.5, 1.3])]:
    sol = newton_sistemas(guess, F, J, tol=1e-12)
    print(sol)


[-1.20854348 -1.02166742]
[2.78711154 1.38079625]


## Atividade 4
Uma indústria consome energia elétrica de três usinas fornecedoras.  

O custo de fornecimento em reais por hora como função da potência consumida em kW é dada pelas seguintes funções:

$$
C_{1}(x) = 10 + 0.3x + 10^{-4}x^{2} + 3.4 \cdot 10^{-9}x^{4} \tag{5.61}
$$

$$
C_{2}(x) = 50 + 0.25x + 2 \cdot 10^{-4}x^{2} + 4.3 \cdot 10^{-7}x^{3}
$$

$$
C_{3}(x) = 500 + 0.19x + 5 \cdot 10^{-4}x^{2} + 1.1 \cdot 10^{-7}x^{4}
$$

Calcule a distribuição de consumo que produz custo mínimo quando a potência total consumida é **1500 kW**.  

Denote por $x_{1}, x_{2}, x_{3}$ as potências consumidas das usinas 1, 2 e 3, respectivamente.  

O custo total será dado por:

$$
C(x_{1},x_{2},x_{3}) = C_{1}(x_{1}) + C_{2}(x_{2}) + C_{3}(x_{3})
$$

Enquanto o consumo total é:

$$
G(x_{1},x_{2},x_{3}) = x_{1} + x_{2} + x_{3} - 1500 = 0
$$

Pelos multiplicadores de Lagrange, temos que resolver o sistema:

$$
\nabla C(x_{1},x_{2},x_{3}) = \lambda \nabla G(x_{1},x_{2},x_{3})
$$

$$
G(x_{1},x_{2},x_{3}) = 0
$$

**Resposta esperada:**

$$
(x_{1},x_{2},x_{3}) \approx (453.62, \; 901.94, \; 144.43)
$$

In [7]:
import numpy as np
from utils.algoritmos import newton_sistemas


def F(v):
    x1, x2, x3, lam = v
    return np.array(
        [
            0.3 + 2.0e-4 * x1 + 13.6e-9 * x1 ** 3 - lam,
            0.25 + 4.0e-4 * x2 + 1.29e-6 * x2 ** 2 - lam,
            0.19 + 1.0e-3 * x3 + 4.4e-7 * x3 ** 3 - lam,
            x1 + x2 + x3 - 1500.0,
        ],
        dtype=float,
    )


def J(v):
    x1, x2, x3, lam = v
    return np.array(
        [
            [2.0e-4 + 40.8e-9 * x1 ** 2, 0.0, 0.0, -1.0],
            [0.0, 4.0e-4 + 2.58e-6 * x2, 0.0, -1.0],
            [0.0, 0.0, 1.0e-3 + 1.32e-6 * x3 ** 2, -1.0],
            [1.0, 1.0, 1.0, 0.0],
        ],
        dtype=float,
    )


x0 = np.array([500.0, 500.0, 500.0, 0.3], dtype=float)
sol = newton_sistemas(x0, F, J, tol=1e-12)
print(f"x1 ≈ {sol[0]:.2f}, x2 ≈ {sol[1]:.2f}, x3 ≈ {sol[2]:.2f}")
print(sol)


x1 ≈ 453.62, x2 ≈ 901.94, x3 ≈ 144.43
[453.62238916 901.94318404 144.4344268    1.66019422]


## Versionando o código

Submeta a branch para o servidor:

```bash
git add .
git commit -m "Semana 7"
git push origin semana7
```